In [2]:
!pip install -q faiss-cpu sentence-transformers transformers torch pandas numpy scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 66.1 MB/s eta 0:00:00


In [3]:
from google.colab import files
uploaded = files.upload()

Saving clean_reviews.csv to clean_reviews.csv


In [4]:
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from transformers import pipeline
from datetime import datetime
from sklearn.preprocessing import MinMaxScaler

In [ ]:
df = pd.read_csv("clean_reviews.csv")

required_cols = ["review", "rating"]
assert all(col in df.columns for col in required_cols)

df = df.dropna(subset=["review"])
df.head()

,product,review,rating
0,Kindle Paperwhite,I initially had trouble deciding between the p...,5.0
1,Kindle Paperwhite,Allow me to preface this with a little history...,5.0
2,Kindle Paperwhite,I am enjoying it so far. Great for reading. Ha...,4.0
3,Kindle Paperwhite,I bought one of the first Paperwhites and have...,5.0
4,Kindle Paperwhite,I have to say upfront - I don't like coroporat...,5.0


In [ ]:
def chunk_text(text, chunk_size=350):
    words=text.split()
    return [
        " ".join(words[i:i +chunk_size])
        for i in range(0, len(words), chunk_size)]

chunks, metadata = [], []
for _, row in df.iterrows():
    for chunk in chunk_text(row["review"]):
        chunks.append(chunk)
        metadata.append({
            "rating": row["rating"]
        })

print(f"Total chunks created: {len(chunks)}")

Total chunks created: 1063


In [ ]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embedder.encode(chunks, show_progress_bar=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/34 [00:00<?, ?it/s]

In [ ]:
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))
print("FAISS vectors stored:", index.ntotal)

FAISS vectors stored: 1063


In [ ]:
def retrieve_reviews(
    query,
    top_k=10,
    min_rating=None
):
    query_vec = embedder.encode([query])
    distances, indices = index.search(query_vec, top_k * 3)

    results = []

    for idx in indices[0]:
        meta = metadata[idx]

        if min_rating and meta["rating"] < min_rating:
            continue

        results.append({
            "review": chunks[idx],
            "metadata": meta
        })

        if len(results) >= top_k:
            break

    return results

In [ ]:
def sentiment_score(rating):
    if rating>=4:
        return 1
    elif rating==3:
        return 0
    else:
        return -1

In [ ]:
def trend_weight(date):
    days_old=(datetime.now()-date).days
    return np.exp(-days_old/180)

In [ ]:
generator = pipeline(
    "text-generation",
    model="google/flan-t5-base",
    max_new_tokens=300

)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaF

In [ ]:
def rag_pipeline(
    query
):
    retrieved = retrieve_reviews(
        query,
        top_k=6
    )

    trend_score = 0
    context_blocks = []

    for r in retrieved:
        s = sentiment_score(r["metadata"]["rating"])
        # Removed trend_weight and date related logic as 'date' is not in metadata
        # w = trend_weight(r["metadata"]["date"])
        # trend_score += s * w

        context_blocks.append(
            f"- {r['review']} (Rating: {r['metadata']['rating']})"
        )

    # Trend direction based on sentiment score, assuming a simple aggregation without dates
    trend_direction = (
        "Positive Market Momentum" if trend_score > 0
        else "Negative Market Momentum"
    )

    prompt = f"""
You are an AI market intelligence system.

Context:
{chr(10).join(context_blocks)}

Question:
{query}

Tasks:
1. Identify dominant consumer sentiment
2. Extract recurring pain points
3. Determine market trend direction
4. Justify conclusions using the context

Market Trend Signal:
{trend_direction}
"""

    return generator(prompt)[0]["generated_text"]

In [ ]:
print(rag_pipeline(
        query="Why are customers unhappy with electronics products?")
)

Token indices sequence length is longer than the specified maximum sequence length for this model (1582 > 512). Running this sequence through the model will result in indexing errors
Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



You are an AI market intelligence system.

Context:
- Having worked in the electronics retail industry for years now, I've seen scores of Smart devices come and go. Until now, nobody quite got it right. In the Echo Dot, Amazon has created a near perfect blend of hardware and software. I've seen plenty of the former, but truly seamless multi platform software has eluded everyone but Amazon. We're talking major players like Samsung and Google who have been at it for much longer than Amazon. The main problem is that excellent products like the Samsung Smart Things hub, which do a fantastic job of unifying a slew of different connected devices from different companies (Nest, Honeywell, Phillips, and so on), still lacked the web connectivity and entertainment support I wanted, so I'd still end up needing my tablet or phone. Thanks to fantastic third party support, the Dot has no problem controlling all of my smart stuff while allowing me to listen to music, order food, check the weather, l